# Zadaca 2 - Imad Buljic

U ovoj zadaci je implementiran faktorizacijski model za recommendation sistem.
Koristi se MovieLens `ratings.csv`, prikazuje se skup podataka, sparse matrica
korisnik-film, te rekonstruisana matrica nakon faktorizacije.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")


## Ucitavanje skupa podataka

Koristi se `ratings.csv` iz MovieLens skupa. Svaki red sadrzi korisnika, film,
ocjenu i timestamp. Za pregled i kasniju vizualizaciju koristi se manji podskup
najaktivnijih korisnika i najocjenjivanijih filmova.


In [ ]:
df_ratings = pd.read_csv("ratings.csv")

print(f"Ukupno ocjena: {len(df_ratings)}")
print(f"Broj korisnika: {df_ratings['userId'].nunique()}")
print(f"Broj filmova: {df_ratings['movieId'].nunique()}")
print(f"Raspon ocjena: {df_ratings['rating'].min()} - {df_ratings['rating'].max()}")
print()

display(df_ratings.head(10))


## Formiranje matrice korisnik-film

Da bi matrica bila pregledna, uzima se 15 najaktivnijih korisnika i 15 filmova koji
globalno imaju najvise ocjena. Pivot tabela predstavlja originalnu matricu sa
poznatim ocjenama i praznim poljima tamo gdje korisnik nije ocijenio film.


In [ ]:
top_korisnici = df_ratings["userId"].value_counts().head(15).index.tolist()
top_filmovi = df_ratings["movieId"].value_counts().head(15).index.tolist()

df_sub = df_ratings[
    df_ratings["userId"].isin(top_korisnici) & df_ratings["movieId"].isin(top_filmovi)
].copy()

matrica_df = df_sub.pivot_table(index="userId", columns="movieId", values="rating")
matrica_original = matrica_df.fillna(0)
R = matrica_original.values.astype(float)

korisnici = [f"User {uid}" for uid in matrica_original.index]
filmovi = [f"Film {mid}" for mid in matrica_original.columns]

print(f"Dimenzije matrice: {matrica_original.shape}")
print(f"Ukupno polja: {matrica_original.size}")
print(f"Poznatih ocjena: {np.count_nonzero(R)}")
print(f"Praznih polja: {matrica_original.size - np.count_nonzero(R)}")
print(f"Gustoca matrice: {np.count_nonzero(R) / matrica_original.size:.1%}")
print()

display(matrica_df)


## Sparse matrica

Originalna matrica se pretvara u CSR sparse format. Tako se jasno vidi da veliki broj
polja nema ocjenu i da je matrica suplja. Ovo je bitan korak prije faktorizacije.


In [ ]:
R_sparse = csr_matrix(R)

print("CSR sparse matrica:")
print(R_sparse)
print()
print(f"Oblik: {R_sparse.shape}")
print(f"Broj nenultih elemenata: {R_sparse.nnz}")
print(f"Sparsity: {1 - R_sparse.nnz / (R_sparse.shape[0] * R_sparse.shape[1]):.1%}")


In [ ]:
sparse_prikaz = R.copy()
sparse_prikaz[sparse_prikaz == 0] = np.nan

plt.figure(figsize=(10, 5))
plt.imshow(sparse_prikaz, cmap="YlGnBu", aspect="auto", vmin=0.5, vmax=5)
plt.colorbar(label="Ocjena")
plt.xticks(range(len(filmovi)), filmovi, rotation=45, ha="right")
plt.yticks(range(len(korisnici)), korisnici)
plt.title("Originalna sparse matrica ocjena")

for i in range(len(korisnici)):
    for j in range(len(filmovi)):
        if R[i, j] == 0:
            plt.text(j, i, "?", ha="center", va="center", color="gray", fontsize=8)
        else:
            plt.text(j, i, f"{R[i, j]:.1f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()


## Faktorizacija pomocu TruncatedSVD

Ovdje se koristi `TruncatedSVD` kao faktorizacijski model. Ideja je da se velika i
suplja matrica aproksimira proizvodom dvije manje matrice. Nakon toga se moze dobiti
rekonstruisana matrica sa procijenjenim vrijednostima i za ranije prazna polja.


In [ ]:
broj_komponenti = 5
svd_model = TruncatedSVD(n_components=broj_komponenti, random_state=42)

U_sigma = svd_model.fit_transform(R_sparse)
VT = svd_model.components_
R_pred_sirovo = U_sigma @ VT
R_pred = np.clip(R_pred_sirovo, 0.5, 5.0)

print(f"Originalna matrica R: {R.shape}")
print(f"Matrica korisnik-komponente: {U_sigma.shape}")
print(f"Matrica komponente-film: {VT.shape}")
print(f"Explained variance ratio: {svd_model.explained_variance_ratio_.sum():.4f}")


In [ ]:
U_df = pd.DataFrame(
    U_sigma,
    index=korisnici,
    columns=[f"Komponenta {i+1}" for i in range(U_sigma.shape[1])]
)
VT_df = pd.DataFrame(
    VT,
    index=[f"Komponenta {i+1}" for i in range(VT.shape[0])],
    columns=filmovi
)

print("Matrica korisnik-komponente:")
display(U_df.round(3))

print("Matrica komponente-film:")
display(VT_df.round(3))


## Rekonstruisana matrica poslije faktorizacije

Poslije faktorizacije dobija se nova matrica sa procijenjenim ocjenama za sva polja.
Tu se sada moze vidjeti kako izgledaju vrijednosti i na mjestima gdje su ranije bila
prazna polja u sparse matrici.


In [ ]:
R_pred_df = pd.DataFrame(R_pred, index=korisnici, columns=filmovi)

print("Rekonstruisana matrica:")
display(R_pred_df.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].imshow(sparse_prikaz, cmap="YlGnBu", aspect="auto", vmin=0.5, vmax=5)
axes[0].set_title("Prije: sparse matrica")
axes[0].set_xticks(range(len(filmovi)))
axes[0].set_xticklabels(filmovi, rotation=45, ha="right", fontsize=8)
axes[0].set_yticks(range(len(korisnici)))
axes[0].set_yticklabels(korisnici, fontsize=8)

axes[1].imshow(R_pred, cmap="YlGnBu", aspect="auto", vmin=0.5, vmax=5)
axes[1].set_title("Poslije: rekonstruisana matrica")
axes[1].set_xticks(range(len(filmovi)))
axes[1].set_xticklabels(filmovi, rotation=45, ha="right", fontsize=8)
axes[1].set_yticks(range(len(korisnici)))
axes[1].set_yticklabels(korisnici, fontsize=8)

for i in range(len(korisnici)):
    for j in range(len(filmovi)):
        if R[i, j] == 0:
            axes[1].text(j, i, f"{R_pred[i, j]:.1f}", ha="center", va="center", color="darkblue", fontsize=7)
        else:
            axes[1].text(j, i, f"{R_pred[i, j]:.1f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()


## Primjer preporuke

Za jednog korisnika se izdvajaju filmovi koje jos nije ocijenio, a zatim se prikazuju
oni sa najvisom procijenjenom ocjenom iz rekonstruisane matrice.


In [ ]:
kandidati = []
for red in range(R.shape[0]):
    indeksi = np.where(R[red] == 0)[0]
    if len(indeksi) == 0:
        continue
    najbolji_sirovi_score = float(np.max(R_pred_sirovo[red, indeksi]))
    kandidati.append((red, najbolji_sirovi_score))

ciljni_red = max(kandidati, key=lambda x: x[1])[0]
ciljni_korisnik = korisnici[ciljni_red]

neocijenjeni = np.where(R[ciljni_red] == 0)[0]
preporuke = []

for idx in neocijenjeni:
    preporuke.append((filmovi[idx], R_pred_sirovo[ciljni_red, idx], R_pred[ciljni_red, idx]))

preporuke_df = pd.DataFrame(preporuke, columns=["Film", "Sirovi score", "Predvidjena ocjena"])
preporuke_df = preporuke_df.sort_values("Sirovi score", ascending=False).head(5)

print(f"Top preporuke za {ciljni_korisnik}:")
display(preporuke_df.round(2))


In [ ]:
maska = R > 0
rmse = np.sqrt(np.mean((R_pred[maska] - R[maska]) ** 2))
mae = np.mean(np.abs(R_pred[maska] - R[maska]))

print(f"RMSE nad poznatim ocjenama: {rmse:.4f}")
print(f"MAE nad poznatim ocjenama: {mae:.4f}")


## Zakljucak

U ovoj zadaci je prikazan kompletan tok rada za recommendation problem:
- ucitan je skup podataka
- formirana je korisnik-film matrica
- prikazana je sparse matrica prije faktorizacije
- primijenjen je faktorizacijski model `TruncatedSVD`
- prikazana je rekonstruisana matrica poslije faktorizacije
- izdvojene su preporuke za jednog korisnika

Time su ispunjeni trazeni elementi zadatka: vidi se skup podataka, sparse matrica i
matrica poslije faktorizacije.
